# vcc_2026_8 — experiments after config 1

**Reference point:** config 1 as recorded in `vcc_2026_7.ipynb` (record cell) and `results/7/config1_record.json`. Every experiment here reports against it and against C+G, on Eval A (H1 held out).

**Rules carried over:**
- `code/cellgen/cell_generator.py` and `scripts/harness/` stay frozen; import them, never copy or edit them.
- No `vcc submit` without Jose's explicit OK (the last entry counts; deadline Nov 5, 23:59 UTC).
- vcc2027 is started only for jobs Jose assigns to it, and stopped when idle.
- Outputs go to `results/v8/<experiment>/`.

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

V = Path.home() / "vcc"
for p in (V / "code/vcc6", V / "scripts", V / "scripts/harness", V / "code/cellgen"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

OUT = V / "results/v8"; OUT.mkdir(parents=True, exist_ok=True)
REF = json.loads((V / "results/7/config1_record.json").read_text())
print("config 1:", REF["settings"] | {"frozen": REF["frozen"]})
display(pd.DataFrame(REF["evalA_H1_heldout_scaled"]).loc[["mse", "pds", "nmae", "fidelity", "reach", "jaccard", "overall"]].round(4))

## Experiments
One section per experiment: the question, the arm(s), the output folder, then the result against the reference table above.

### Step 15 — config 1′, knockdown strength, end to end (Jose, 2026-10-30)
**Write-up:** `results/stage1/2026-10-28_scorer_space/report.md` (section "Step 15"; all research goes in that one report). **Outputs:** `results/stage1/2026-10-29_step15/` (tables/, logs/). **Code:** `code/vcc6/step15.py` (new module; the frozen generator, harness and config-1 files are imported, not changed).

**Config 1′** (F0): called genes 0.5 × C+G LFC; non-called 0.6 × c × Δ_pool, c = z_a²/(z_a² + 1); single-source c₁ = 1; no source → control. Config 1 is the same with 0.5 on the non-called side.
**Calls:** K0 = C+G's calls; K4 1,200 = per knockdown the top 1,200 tested genes by C+G z (magnitudes unchanged; the magnitude split stays K0's).

**15a / 15b** run on vcc2027 with the noise-free proxy only (no cells):
```
~/venv/bin/python code/vcc6/step15.py step15a
~/venv/bin/python code/vcc6/step15.py oracle --part P --nparts N        # P = 0..N-1, one process each
~/venv/bin/python code/vcc6/step15.py predictors
~/venv/bin/python code/vcc6/step15.py predrun --pred b --part P --nparts 8         # (b) = best held-out predictor
~/venv/bin/python code/vcc6/step15.py o3 [--pred b] --part P --nparts 8          # addendum O3
~/venv/bin/python code/vcc6/step15.py lamext --pred b                            # side check, lambda 3/4/6
~/venv/bin/python code/vcc6/step15.py fit
```
Run 2026-10-30 03:01–04:09 UTC on vcc2027 (`logs/`). The cells below only read the tables; the write-up is the Step 15 section of the scorer-space report.md. **15c** (cells) waits for Jose's go.

In [ ]:
S15 = V / "results/stage1/2026-10-29_step15/tables"
COMP7 = ["overall", "mse", "pds", "nmae", "fidelity", "reach", "jaccard"]
if (S15 / "step15a.csv").exists():
    t15a = pd.read_csv(S15 / "step15a.csv")
    display(t15a.set_index(["line", "part", "config", "calls"])[COMP7 + ["calls_median"]].round(4))

In [ ]:
if (S15 / "step15b.json").exists():
    b = json.loads((S15 / "step15b.json").read_text())
    for k, v in b.items():
        print(k); display(pd.DataFrame(v["heldout_mean"]).T[COMP7].round(4))
if (S15 / "step15b_predictors.json").exists():
    pr = json.loads((S15 / "step15b_predictors.json").read_text())
    display(pd.DataFrame(pr["rho"]).T.round(3)); print("best held-out predictor:", pr["best"])

### Step 15c — config 1′ on cells (Jose's go 2026-10-30)
**Code:** `code/vcc6/step15c.py` (model / boot / diag / compare) and `code/vcc6/step15c.sh` (one chain per VM). vcc_2026_7's data-loader and model cells are copied into `step15c.model` with config 1′; the frozen generator, harness, `run_line.py`, `boot_score.py` and vcc_2026_7 are not changed. **Outputs:** `results/stage1/2026-10-29_step15/c/<config>/<line>/`, comparison tables in `c/tables/`.

One config per VM, each chain strictly in sequence (model's table load → generate → score → boot → diag):
```
bash code/vcc6/step15c.sh config1p_K0 4          # vcc2026
bash code/vcc6/step15c.sh config1p_K4_1200 8     # vcc2027
~/venv/bin/python code/vcc6/step15c.py compare   # vcc2026, after copying vcc2027's harness folders and diag.json
```
Lines: Eval A (H1 held out) and HCT116final (HCT116 out, HEK293T votes). The checks on Eval A: m 0.5 reproduces vcc_2026_7's freeze-check input exactly; calls, LFC and SE equal the scorer-space fold cache.

In [ ]:
C15 = V / "results/stage1/2026-10-29_step15/c"
if (C15 / "tables/step15c_paired.csv").exists():
    pc = pd.read_csv(C15 / "tables/step15c_paired.csv")
    display(pc[pc.metric.isin(COMP7)].pivot_table(index=["line", "part", "cand", "ref"], columns="metric", values="diff")[COMP7].round(4))
    print(json.dumps(json.loads((C15 / "tables/step15c.json").read_text())["adoption"], indent=1))

### Step 17 — volume-matched call allocation by predictor (b) (Jose, 2026-10-30)
Noise-free proxy only, on vcc2027 after its 15c job. **Code:** `code/vcc6/step17.py`, `code/vcc6/step17.sh`. **Outputs:** `results/stage1/2026-10-30_step17/` (tables/, logs/).
Config 1′ magnitudes; calls = top N_k by C+G z. (b)-allocation N_k = clip(λ × q(pct_k), 0, 4,000), with pct_k = (b)'s panel percentile and q = the fit half's (or, volume-free, H1's full) true-count quantiles. Each row is paired with flat K4 at the same median volume; oracle rows use λ × the true count.
```
~/venv/bin/python code/vcc6/step17.py predictor                        # 1c, 4; writes h1_truth.json first
~/venv/bin/python code/vcc6/step17.py rows --line H1 --part P --nparts 4  # and --line K562 / HCT116 / HEK293T
~/venv/bin/python code/vcc6/step17.py summary
```

In [ ]:
S17 = V / "results/stage1/2026-10-30_step17/tables"
for k in ("item1_fitted", "item1b_share", "item2", "item3"):
    if (S17 / f"{k}.csv").exists():
        print(k); display(pd.read_csv(S17 / f"{k}.csv").round(4))

### Step 18 — D/E/F file, config 1′ + K0 (Jose's choice, 2026-10-30; not uploaded)
**Code:** `code/vcc6/step18.py` (model / stats / final), `step18_gen.sh` (generation per VM), `step18_final.sh` + `step18_prep.sh` (swap, merge, `vcc prep`). **Outputs:** `results/stage1/2026-10-30_step18/`; the file is `final/prediction_DEF.vcc` (3.18 GB, `vcc prep` exit 0). Write-up: report section 10.
```
~/venv/bin/python code/vcc6/step18.py model        # D/E/F in one process, all 117 tables, checks vs the uploaded C+G inputs
~/venv/bin/python code/vcc6/step18.py stats        # requested calls and |LFC| vs the uploaded C+G file
bash code/vcc6/step18_gen.sh 8 D E                 # vcc2027;  bash code/vcc6/step18_gen.sh 4 F on vcc2026
bash code/vcc6/step18_final.sh                     # swap, submission files, on-disk merge, checks, vcc prep
```
`vcc prep` needs ~19 GB of free disk for its temp copy as well as ~44 GB of RAM + swap; size the swapfile so both fit.

In [ ]:
S18 = V / "results/stage1/2026-10-30_step18/tables"
for k in ("pre_generation", "final"):
    if (S18 / f"{k}.json").exists():
        print(k); print(json.dumps(json.loads((S18 / f"{k}.json").read_text()), indent=1)[:3000])